In [1]:
from sklearn.datasets import load_iris
import pandas as pd
import numpy as np

In [2]:
iris = load_iris()

In [3]:
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df['target'] = iris.target
df.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [5]:
train_df,test_df = train_test_split(df,test_size=0.2,random_state=42,stratify=df['target'])

In [6]:
X_train = train_df.drop('target', axis=1)
y_train = train_df['target']

X_test = test_df.drop('target', axis=1)
y_test = test_df['target']

In [7]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [8]:
X_train[0]

array([-1.72156775, -0.33210111, -1.34572231, -1.32327558])

In [9]:
X_train_tensor = torch.tensor(X_train,dtype=torch.float32)
X_test_tensor = torch.tensor(X_test,dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values,dtype=torch.long)
y_test_tensor = torch.tensor(y_test.values,dtype=torch.long)

In [10]:
class IrisClassifier(nn.Module):
    
    def __init__(self,input_size,hidden_size,output_size):
        super(IrisClassifier,self).__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size,hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size,hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size,output_size),
        )

    def forward(self,x):
        return self.network(x)

In [11]:
input_size = X_train_tensor.shape[1]
hidden_size = 16
output_size = 3



In [12]:
model = IrisClassifier(input_size,hidden_size,output_size)

In [13]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.01)

In [14]:
## Model Training
epoch = 500
for e in range(epoch):
    model.train()
    optimizer.zero_grad()
    
    predictions = model(X_train_tensor)
    loss=criterion(predictions,y_train_tensor)

    loss.backward()
    optimizer.step()

    if(e+1)%10==0:
        print(f"Epoch [{e+1}] / {epoch} , loss : {loss.item():.4f}")

Epoch [10] / 500 , loss : 0.8440
Epoch [20] / 500 , loss : 0.3978
Epoch [30] / 500 , loss : 0.1384
Epoch [40] / 500 , loss : 0.0700
Epoch [50] / 500 , loss : 0.0489
Epoch [60] / 500 , loss : 0.0411
Epoch [70] / 500 , loss : 0.0369
Epoch [80] / 500 , loss : 0.0346
Epoch [90] / 500 , loss : 0.0327
Epoch [100] / 500 , loss : 0.0310
Epoch [110] / 500 , loss : 0.0294
Epoch [120] / 500 , loss : 0.0279
Epoch [130] / 500 , loss : 0.0266
Epoch [140] / 500 , loss : 0.0255
Epoch [150] / 500 , loss : 0.0244
Epoch [160] / 500 , loss : 0.0232
Epoch [170] / 500 , loss : 0.0217
Epoch [180] / 500 , loss : 0.0200
Epoch [190] / 500 , loss : 0.0178
Epoch [200] / 500 , loss : 0.0154
Epoch [210] / 500 , loss : 0.0124
Epoch [220] / 500 , loss : 0.0094
Epoch [230] / 500 , loss : 0.0067
Epoch [240] / 500 , loss : 0.0048
Epoch [250] / 500 , loss : 0.0036
Epoch [260] / 500 , loss : 0.0027
Epoch [270] / 500 , loss : 0.0022
Epoch [280] / 500 , loss : 0.0018
Epoch [290] / 500 , loss : 0.0015
Epoch [300] / 500 , los

In [15]:
model.eval()
with torch.no_grad():
    y_pred = model(X_test_tensor)
    y_pred_labels = torch.argmax(y_pred,dim=1) ## Class with highest prob

    accuracy = (y_pred_labels == y_test_tensor).sum().item() / y_test_tensor.size(0)

    print(f"Test Acuuracy is : {accuracy}")

Test Acuuracy is : 0.9666666666666667


In [16]:
def predict_iris(sepal_length,sepal_width,petal_length,petal_width):
    input_data = np.array([[sepal_length,sepal_width,petal_length,petal_width]])
    input_data = scaler.transform(input_data)
    input_data = torch.tensor(input_data, dtype=torch.float32)

    model.eval()
    with torch.no_grad():
        prediction = model(input_data)
        prediction_label = torch.argmax(prediction,dim=1).item()
    return prediction_label

In [17]:
predict_iris(5.1,3.5,1.4,0.2)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


0